# Step 00 — Data provenance and objective reproducibility audit

This notebook validates the implemented **step 00** pipeline against the local repository data.

Scope of this step:

- discover the historical SQLite studies and trace sources;
- recompute historical best-trial objectives against candidate trace sources;
- keep the Control provenance ambiguity explicit;
- inventory the 37 ATF files by **region** (`DH`, `VH`) and **condition**.

This step is **provenance-only**. It does not claim biological degeneracy.

In [ ]:
from pathlib import Path
import os
import sys

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path(os.environ.get('ASTROMODEL_PROJECT_ROOT', Path.cwd())).resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.provenance import run_step00_provenance

print(f'PROJECT_ROOT={PROJECT_ROOT}')
print(f'initial_fit_dir={(PROJECT_ROOT / "data" / "1_Initial_xp_fit")}')
print(f'atf_dir={(PROJECT_ROOT / "data" / "2_K+ Pumps Data")}')

In [ ]:
results = run_step00_provenance(PROJECT_ROOT)
db_summary = results['db_study_summary']
trace_summary = results['trace_source_summary']
provenance = results['control_trace_verification']
atf_inventory = results['atf_region_condition_inventory']
atf_counts = results['atf_region_condition_counts']

print('written outputs:', sorted((PROJECT_ROOT / 'outputs' / 'provenance').glob('*.csv')))
print('db rows:', len(db_summary), 'trace rows:', len(trace_summary), 'ATF rows:', len(atf_inventory))

## Historical DB study inventory

In [ ]:
display(db_summary)

pivot_trials = db_summary.pivot(index='current_na', columns='condition', values='n_trials').sort_index()
ax = pivot_trials.plot(marker='o', figsize=(8, 4), title='Historical Optuna trial counts by condition/current')
ax.set_xlabel('Current (nA)')
ax.set_ylabel('Number of trials')
plt.tight_layout()
plt.show()

pivot_best = db_summary.pivot(index='current_na', columns='condition', values='best_objective').sort_index()
ax = pivot_best.plot(marker='o', figsize=(8, 4), title='Best stored objective by condition/current')
ax.set_xlabel('Current (nA)')
ax.set_ylabel('Best stored objective')
plt.tight_layout()
plt.show()

## Trace source inventory

In [ ]:
display(trace_summary)

trace_dt = trace_summary[['trace_source', 'condition', 'rows', 'dt_ms', 'uses_header']].copy()
display(trace_dt)

## Objective reproducibility and Control provenance

In [ ]:
display(provenance)

chosen = provenance.drop_duplicates('db_name')[['db_name', 'condition', 'current_na', 'chosen_trace_source', 'chosen_relative_objective_error', 'chosen_status']].sort_values(['condition', 'current_na'])
display(chosen)

plot_df = provenance.copy()
plot_df['label'] = plot_df['db_name'] + '\n' + plot_df['trace_source']
ax = plot_df.plot(kind='bar', x='label', y='relative_objective_error', figsize=(12, 4), legend=False, title='Relative objective error by DB and candidate trace source')
ax.set_xlabel('DB / candidate trace source')
ax.set_ylabel('Relative objective error')
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()

## Region-aware ATF inventory

In [ ]:
display(atf_inventory.head(10))
display(atf_counts)

counts_pivot = atf_counts.pivot(index='region', columns='condition', values='n_cells').loc[['DH', 'VH']]
ax = counts_pivot.plot(kind='bar', figsize=(8, 4), title='ATF cell counts by region and condition')
ax.set_xlabel('Region')
ax.set_ylabel('Number of cells')
plt.tight_layout()
plt.show()

## Interpretation

This notebook shows that the historical studies are locally auditable and that the 37 ATF files satisfy the updated region-aware design contract. It also shows that the Control historical provenance remains unresolved under the documented objective-recomputation contract, so Control historical fits must not be treated as fully validated reviewer-facing inputs.